# 03. 심화: RL 정책 변화와 pass@k

목표: 논문이 말한 ground-truth amplification, tail discovery, wrong-mode amplification을 toy policy로 구분하고, pass@1과 pass@k가 왜 다르게 움직이는지 확인합니다.

실행 방법: 모든 셀을 순서대로 실행합니다. 외부 패키지는 필요하지 않습니다.

## 1. 정책 분포 도구 만들기

정책은 가능한 move와 확률의 딕셔너리로 표현합니다. 실제 논문은 legal move set 전체와 reasoning trace marginalization을 사용하지만, 여기서는 핵심 분류만 재현합니다.

In [ ]:
def normalize(scores):
    total = sum(scores.values())
    return {move: value / total for move, value in scores.items()}


def top_k(policy, k=3):
    return [move for move, _prob in sorted(policy.items(), key=lambda item: item[1], reverse=True)[:k]]


def classify_change(sft_policy, rl_policy, gold_move, k=3, tail_epsilon=0.05):
    sft_top = set(top_k(sft_policy, k))
    rl_top = set(top_k(rl_policy, k))
    sft_gold_prob = sft_policy.get(gold_move, 0.0)

    if gold_move in sft_top and gold_move in rl_top and rl_policy[gold_move] > sft_policy[gold_move]:
        return "ground_truth_amplification"
    if sft_gold_prob <= tail_epsilon and gold_move in rl_top:
        return "tail_discovery"
    if gold_move not in rl_top and top_k(sft_policy, 1)[0] == top_k(rl_policy, 1)[0]:
        return "wrong_mode_amplification"
    return "mixed_or_other"


examples = {
    "easy": {
        "gold": "Qg7#",
        "sft": normalize({"Qg7#": 0.45, "Qe7": 0.25, "Rb8": 0.20, "Kh2": 0.10}),
        "rl": normalize({"Qg7#": 0.75, "Qe7": 0.10, "Rb8": 0.10, "Kh2": 0.05}),
    },
    "hard_tail": {
        "gold": "Nc6+",
        "sft": normalize({"Qh5": 0.40, "Bd3": 0.30, "Re1": 0.25, "Nc6+": 0.05}),
        "rl": normalize({"Nc6+": 0.45, "Qh5": 0.25, "Bd3": 0.20, "Re1": 0.10}),
    },
    "hard_wrong": {
        "gold": "Bxf7+",
        "sft": normalize({"Qh5": 0.50, "Bc4": 0.25, "Nf3": 0.20, "Bxf7+": 0.05}),
        "rl": normalize({"Qh5": 0.70, "Bc4": 0.15, "Nf3": 0.10, "Bxf7+": 0.05}),
    },
}

for name, item in examples.items():
    label = classify_change(item["sft"], item["rl"], item["gold"])
    print(f"{name:10s} -> {label}")

## 2. pass@1과 pass@k 계산

확률 분포에서 독립적으로 k번 샘플링한다고 단순화하면, gold move가 한 번 이상 나올 확률은 `1 - (1 - p_gold)^k`입니다. 정책이 한 후보에 과하게 집중하면 pass@1은 오르지만 pass@k의 폭넓은 coverage는 기대만큼 오르지 않을 수 있습니다.

In [ ]:
def pass_at_k(policy, gold_move, k):
    p = policy.get(gold_move, 0.0)
    return 1 - (1 - p) ** k


print("case | stage | p_gold | pass@1 | pass@4 | top1")
print("--- | --- | --- | --- | --- | ---")
for name, item in examples.items():
    for stage in ["sft", "rl"]:
        policy = item[stage]
        gold = item["gold"]
        print(
            f"{name} | {stage} | {policy[gold]:.2f} | "
            f"{pass_at_k(policy, gold, 1):.2f} | {pass_at_k(policy, gold, 4):.2f} | {top_k(policy, 1)[0]}"
        )

## 3. 여러 퍼즐 난이도에서의 요약

난이도가 올라가면 gold move가 초기 SFT 분포의 top-k 밖에 있을 가능성이 커집니다. 이때 RL은 정답을 발견할 수도 있지만, 틀린 mode를 더 강화할 위험도 커집니다.

In [ ]:
puzzles = [
    ("B1", examples["easy"]),
    ("B2", examples["easy"]),
    ("B3", examples["hard_tail"]),
    ("B4", examples["hard_tail"]),
    ("B5", examples["hard_wrong"]),
]

counts = {}
for difficulty, item in puzzles:
    label = classify_change(item["sft"], item["rl"], item["gold"])
    counts[label] = counts.get(label, 0) + 1
    print(f"difficulty={difficulty} label={label}")

print("\nsummary")
for label, count in counts.items():
    print(f"{label}: {count}")

## 4. 실무적 시사점

RL 목적함수가 pass@1만 밀어붙이면 모델은 정답 mode를 강화할 수도 있지만 틀린 mode도 강화할 수 있습니다. 그래서 어려운 reasoning task에서는 verifier, exploration, entropy, 후보 coverage 같은 장치를 함께 설계해야 합니다.